## This notebook is to train a model to make reviews on a movie

In [1]:
import pandas as pd
import numpy as np
import transformers as tr
import torch 

from datasets import load_dataset # loads large datasets from the Hugging Face Hub
from datasets import Dataset # allows for the creation of custom datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments # Load pre-trained models and classification heads
from transformers import AutoModelForSeq2SeqLM # Load text-to-text models
from transformers import Trainer, TrainingArguments # Load the Trainer class and training arguments 
from transformers import pipeline # Load the pipeline class for easy inference


movieData = load_dataset("stanfordnlp/imdb") # Load the IMDB dataset from the Hugging Face Hub

movieDF = pd.DataFrame(movieData['train']) # Convert the training split to a pandas DataFrame
movieDF.head() # Display the first few rows of the DataFrame

c:\Users\santi\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [2]:
movieDF = movieDF.dropna() # Drop any rows with missing values
movieDF = movieDF.reset_index(drop=True) # Reset the index of the DataFrame

movieHug = Dataset.from_pandas(movieDF) # Convert the pandas DataFrame back to a Hugging Face Dataset  
movieDict = movieHug.train_test_split(test_size=0.2) # Split the dataset into training and testing sets 

token = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Load the tokenizer for the DistilBERT model

def tokenize_function(examples):
    return token(examples["text"], padding="max_length", truncation=True) # Tokenize the text data with padding and truncation

tokenizedDatasets = movieDict.map(tokenize_function, batched=True) # Apply the tokenization function to the dataset
tokenizedDatasets = tokenizedDatasets.remove_columns(["text"]) # Remove the original text column
tokenizedDatasets.set_format("torch") # Set the format of the dataset to PyTorch tensors

Map: 100%|██████████| 5000/5000 [00:01<00:00, 4508.95 examples/s]


In [6]:
import evaluate # Load the evaluate library for model evaluation
import accelerate # Load the accelerate library for distributed training
accuracy = evaluate.load("accuracy") # Load the accuracy metric for evaluation

autoModel = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # Load the pre-trained DistilBERT model for sequence classification with 2 labels

def compute_metrics(eval_pred):
    logits, labels = eval_pred # Unpack the evaluation predictions into logits and labels
    predictions = np.argmax(logits, axis=-1) # Get the predicted class by taking the argmax of the logits
    return accuracy.compute(predictions=predictions, references=labels) # Compute and return the accuracy metric

trainingArgs = TrainingArguments(
    output_dir="./results", # Directory to save the model and training results
    eval_strategy="epoch", # Evaluate the model at the end of each epoch
    save_strategy="epoch", # Save the model at the end of each epoch
    learning_rate=2e-5, # Set the learning rate for training
    per_device_train_batch_size=32, # Set the batch size for training
    per_device_eval_batch_size=32, # Set the batch size for evaluation
    num_train_epochs=3, # Set the number of training epochs
    fp16=True, # Enable mixed precision training for faster training and lower memory usage
    logging_steps=10, # Log training progress every 10 steps
    load_best_model_at_end=True, # Load the best model at the end of training based on evaluation metrics
    metric_for_best_model="eval_loss", # Use evaluation loss as the metric to determine the best model
    greater_is_better=False, # Specify that lower evaluation loss is better for model selection
)

trainer = Trainer(model=autoModel, args=trainingArgs, train_dataset=tokenizedDatasets["train"], eval_dataset=tokenizedDatasets["test"], compute_metrics=compute_metrics) # Create a Trainer object for training and evaluation


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12502.02it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
model = trainer.train() # Train the model using the Trainer object

Epoch,Training Loss,Validation Loss,Accuracy
1,0.166455,0.195814,0.925000
2,0.103979,0.253537,0.926600
3,0.040523,0.292392,0.928000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.78it/s]


In [13]:
text = "This movie was awful! I did not enjoy it and would not recommend it to everyone."
inputs = token(text, return_tensors="pt", truncation=True, padding=True).to(trainer.model.device) # Tokenize the input text and convert it to PyTorch tensors

with torch.no_grad(): # Disable gradient calculation for inference
    outputs = trainer.model(**inputs) # Get the model outputs for the input text
    prediction= torch.argmax(outputs.logits, dim=-1).item() # Get the predicted class by taking the argmax of the logits

print("Predicted class:", prediction) # Print the predicted class (0 for negative, 1 for positive)

Predicted class: 0


In [18]:
from transformers import pipeline # Load the pipeline class for easy inference

generator = pipeline(
    "text-generation", # Specify the task as text generation
    model="Qwen/Qwen2.5-1.5B-Instruct", # Use the Qwen model for text generation
    torch_dtype = torch.bfloat16,
    device_map="auto"
)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 350.98it/s]


In [19]:
print(generator)

TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}


In [37]:
movieTitle = "Spider-Man: No Way Home"
messages = [
    {"role" : "user", "content": f"Please write a sentence overview of the plot of {movieTitle}"}
]

result = generator(messages, max_new_tokens=50, do_sample=True, temperature=0.7) # Generate a response from the model based on the input messages

overview = result[0]['generated_text'] # Extract the generated text from the result
print(overview[-1]['content']) # Print the generated overview

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Spider-Man: No Way Home is a superhero film that takes place in the Marvel Cinematic Universe and follows Peter Parker as he tries to stop Doctor Strange from altering the multiverse.


In [29]:
inputs = token(overview[-1]['content'], return_tensors="pt", truncation=True, padding=True).to(trainer.model.device) # Tokenize the input text and convert it to PyTorch tensors

with torch.no_grad(): # Disable gradient calculation for inference
    outputs = trainer.model(**inputs) # Get the model outputs for the input text
    prediction= torch.argmax(outputs.logits, dim=-1).item() # Get the predicted class by taking the argmax of the logits

print("Predicted class:", prediction) # Print the predicted class (0 for negative, 1 for positive)

Predicted class: 1


In [46]:
emotionClassifier = pipeline(
    "text-classification", # Specify the task as text classification
    model="j-hartmann/emotion-english-distilroberta-base", # Use the emotion classification model
    top_k=None
)

c:\Users\santi\Code\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\santi\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 34590.16it/s]


In [56]:
def generate_movie_overview(movie_title):
    # Build the prompt
    messages = [
        {"role": "user", "content": f"Please write a sentence overview of the plot of {movie_title}"}
    ]
    
    # Generate the overview
    result = generator(messages, max_new_tokens=100, do_sample=True, temperature=0.7)
    overview = result[0]["generated_text"]
    overview_text = overview[-1]["content"]
       
    # Run emotion classification on the overview
    emotions = emotionClassifier(overview_text)
    
    # Print results
    print(f"Movie: {movie_title}")
    print(f"Overview: {overview_text}")
    for emotion in emotions[0]:
            print(f"Emotion: {emotion['label']}, Score: {emotion['score']:.4f}")

generate_movie_overview("The Outsiders")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Movie: The Outsiders
Overview: The novel "The Outsiders" is set in 1960s Los Angeles and follows the lives of two rival gangs, the Greasers and the Socs. It explores themes of class conflict, friendship, loyalty, and coming-of-age experiences through the stories of Ponyboy Curtis and his family's involvement with both groups.
Emotion: neutral, Score: 0.8576
Emotion: joy, Score: 0.0585
Emotion: disgust, Score: 0.0352
Emotion: anger, Score: 0.0178
Emotion: sadness, Score: 0.0162
Emotion: fear, Score: 0.0081
Emotion: surprise, Score: 0.0066


In [57]:
## Polishing model

autoModel = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # Load the pre-trained DistilBERT model for sequence classification with 2 labels

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11112.51it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [58]:
trainingArgs = TrainingArguments(
    output_dir="./results", # Directory to save the model and training results
    eval_strategy="epoch", # Evaluate the model at the end of each epoch
    save_strategy="epoch", # Save the model at the end of each epoch
    learning_rate=2e-5, # Set the learning rate for training
    per_device_train_batch_size=32, # Set the batch size for training
    per_device_eval_batch_size=32, # Set the batch size for evaluation
    num_train_epochs=3, # Set the number of training epochs
    fp16=True, # Enable mixed precision training for faster training and lower memory usage
    logging_steps=10, # Log training progress every 10 steps
    load_best_model_at_end=True, # Load the best model at the end of training based on evaluation metrics
    metric_for_best_model="eval_loss", # Use evaluation loss as the metric to determine the best model
    greater_is_better=False, # Specify that lower evaluation loss is better for model selection
)

trainer = Trainer(
    model=autoModel, 
    args=trainingArgs,
    train_dataset=tokenizedDatasets["train"],
    eval_dataset=tokenizedDatasets["test"],
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.190921,0.197855,0.920000
2,0.155445,0.217313,0.926600
3,0.128771,0.258427,0.926000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.77it/s]


TrainOutput(global_step=1875, training_loss=0.17248407983779906, metrics={'train_runtime': 361.4986, 'train_samples_per_second': 165.976, 'train_steps_per_second': 5.187, 'total_flos': 7948043919360000.0, 'train_loss': 0.17248407983779906, 'epoch': 3.0})

In [59]:
print(trainer.state.best_model_checkpoint)

./results\checkpoint-625


In [60]:
trainer.save_model("./my-model-fixed")
token.save_pretrained("./my-model-fixed")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.74it/s]


('./my-model-fixed\\tokenizer_config.json', './my-model-fixed\\tokenizer.json')

In [62]:
def checkSentiment(text):
    inputs = token(text, return_tensors="pt", truncation=True, padding=True).to(trainer.model.device) # Tokenize the input text and convert it to PyTorch tensors

    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = trainer.model(**inputs) # Get the model outputs for the input text
        prediction= torch.argmax(outputs.logits, dim=-1).item() # Get the predicted class by taking the argmax of the logits
        probs = torch.softmax(outputs.logits, dim=-1) # Calculate the probabilities for each class using softmax

    return prediction, probs # Return the predicted class and probabilities (0 for negative, 1 for positive)

print(checkSentiment("This movie was awful! I did not enjoy it and would not recommend it to everyone."))
print(checkSentiment("This movie was amazing! I loved it and would highly recommend it to everyone."))
print(checkSentiment("This movie was okay. It had some good moments, but also some bad ones."))
print(checkSentiment("This movie was okay at best. It was mediocre in some parts, but it was entertaining enough to watch. I would recommend it to some people, but not everyone."))

(0, tensor([[0.9878, 0.0122]], device='cuda:0'))
(1, tensor([[0.0111, 0.9889]], device='cuda:0'))
(0, tensor([[0.5139, 0.4861]], device='cuda:0'))
(1, tensor([[0.2621, 0.7379]], device='cuda:0'))
